# VidTranscribe.ai - End-to-End Evaluation on Colab

Notebook này được thiết kế để chạy toàn bộ pipeline và 3 benchmark CV-ready của dự án VidTranscribe.ai ngay trên Google Colab.

**Các bước tự động:**
1. Mount Google Drive để lưu kết quả
2. Clone project từ GitHub
3. Cài đặt môi trường (FFmpeg, Ollama, Python deps)
4. Chuẩn bị Hybrid Dataset (70% PhoST, 30% Tech/Alphanumeric)
5. Chạy 3 Benchmark: Localization, Performance (VRAM profile), Group Sync
6. Lưu báo cáo về Drive

## 1. Mount Google Drive
Kết nối với Drive để sau khi chạy xong, kết quả `.csv` và `.json` sẽ được tự động copy sang Drive của bạn.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Pull / Clone Repository
Thay đổi `YOUR_GITHUB_URL` thành link repo của bạn để Colab kéo code mới nhất về.

In [ ]:
import os

# THAY ĐỔI URL DƯỚI ĐÂY THÀNH REPO CỦA BẠN
REPO_URL = "https://github.com/HoangKhang226/VidTranscribe.ai.git"

!git clone $REPO_URL /content/VidTranscribe.ai
%cd /content/VidTranscribe.ai

## 3. Cài đặt System Dependencies (FFmpeg & Ollama)

In [ ]:
# Cài đặt FFmpeg cho xử lý audio/video
!apt-get update -y && apt-get install -y ffmpeg

# Cài đặt Ollama Linux
!curl -fsSL https://ollama.com/install.sh | sh

# Chạy Ollama server dưới background
!nohup ollama serve > ollama.log 2>&1 &

# Đợi 5 giây cho server khởi động
import time
time.sleep(5)

# Kéo model Qwen2.5 (như trong cấu hình hiện tại)
!ollama pull qwen2.5:3b-instruct-q4_K_M

## 4. Cài đặt Python Dependencies

In [ ]:
!pip install -r requirements.txt

## 5. Chuẩn bị Hybrid Dataset (70% PhoST, 30% Tech Stress Test)
Đoạn code này sẽ kéo 35 câu từ bộ `vinai/PhoST` trên HuggingFace và kết hợp với 15 câu bẫy kỹ thuật đa ngành để tạo thành bộ `evaluation/test_cases.json` chuẩn.

In [ ]:
from datasets import load_dataset
import json

# 1. Tải 35 câu từ PhoST (Spoken flow)
print("Đang tải PhoST dataset từ HuggingFace...")
ds = load_dataset("vinai/PhoST", "en-vi", split="test")
phost_cases = []
for i in range(35):
    phost_cases.append({
        "id": f"phost_{i:03d}",
        "domain": "speech_flow",
        "input": ds[i]['en'],
        "gold": ds[i]['vi'],
        "predicted": ""
    })

# 2. Khởi tạo 15 câu bẫy kỹ thuật (Multi-domain Tech Stress Test)
tech_cases = [
    {"id": "tech_001", "domain": "medical", "input": "We use mRNA vaccines for COVID-19 prevention.", "gold": "Chúng tôi sử dụng vắc-xin em a ren a để phòng ngừa cô vít mười chín.", "predicted": ""},
    {"id": "tech_002", "domain": "aviation", "input": "The A320 aircraft is ready for FCL cargo.", "gold": "Máy bay a ba hai không đã sẵn sàng cho hàng hóa ép xê lờ.", "predicted": ""},
    {"id": "tech_003", "domain": "finance", "input": "P2P lending with T+2 settlement.", "gold": "Cho vay pi tu pi với thanh toán tê cộng hai.", "predicted": ""},
    {"id": "tech_004", "domain": "it", "input": "Implement OAuth2 with PostgreSQL.", "gold": "Triển khai ô o hai với pốt gờ rê ét qui eo.", "predicted": ""},
    {"id": "tech_005", "domain": "it", "input": "Configure Nginx to serve HTML5.", "gold": "Cấu hình en gin ích để phục vụ ét ch em e lăm.", "predicted": ""},
    {"id": "tech_006", "domain": "finance", "input": "Calculate the EBITDA margin.", "gold": "Tính toán biên lợi nhuận i bít đa.", "predicted": ""},
    {"id": "tech_007", "domain": "it", "input": "Deploy BM25 algorithm on GitHub.", "gold": "Triển khai thuật toán bi em hai lăm trên gít húp.", "predicted": ""},
    {"id": "tech_008", "domain": "it", "input": "Use FFmpeg to convert MP4 to WebP.", "gold": "Sử dụng ép ép em pếch để chuyển đổi em pi pho sang wép pi.", "predicted": ""},
    {"id": "tech_009", "domain": "it", "input": "GraphQL API replaces REST.", "gold": "Gờ ráp qui eo a pi ai thay thế rét.", "predicted": ""},
    {"id": "tech_010", "domain": "finance", "input": "B2B sales integration.", "gold": "Tích hợp bán hàng bi tu bi.", "predicted": ""},
    {"id": "tech_011", "domain": "logistics", "input": "The 4PL logistics provider.", "gold": "Nhà cung cấp dịch vụ logistics four pê e lờ.", "predicted": ""},
    {"id": "tech_012", "domain": "medical", "input": "Install Vitamin B12 injection.", "gold": "Tiêm vi ta min bê mười hai.", "predicted": ""},
    {"id": "tech_013", "domain": "it", "input": "LLM agents running on CUDA.", "gold": "Các tác nhân eo eo em chạy trên cu đa.", "predicted": ""},
    {"id": "tech_014", "domain": "it", "input": "Scale the Kubernetes cluster.", "gold": "Mở rộng cụm cu bơ nê tịt.", "predicted": ""},
    {"id": "tech_015", "domain": "it", "input": "Connect via SSH protocol.", "gold": "Kết nối qua giao thức ét ét hát.", "predicted": ""}
]

hybrid_cases = phost_cases + tech_cases

import os
os.makedirs("evaluation", exist_ok=True)
with open("evaluation/test_cases.json", "w", encoding="utf-8") as f:
    json.dump(hybrid_cases, f, ensure_ascii=False, indent=2)
    
print(f"Đã tạo thành công {len(hybrid_cases)} test cases.")

## 6. Chạy Benchmark 1: Text Localization
Đo lường Token Error Rate trên bộ dataset 50 câu vừa tạo.

In [ ]:
!python evaluation/benchmark_localization.py

## 7. Chạy Benchmark 2: End-to-End Performance & Hardware
Colab có NVIDIA T4 GPU (nếu bạn chọn Runtime > T4 GPU). Pipeline sẽ tận dụng CUDA để chạy Whisper.
Benchmark này tự động đo RTF và Peak VRAM.

In [ ]:
# Tải một video sample siêu nhẹ nếu thư mục chưa có
!wget -q -O evaluation/Download.mp4 https://github.com/intel-iot-devkit/sample-videos/raw/master/person-bicycle-car-detection.mp4

# Chạy pipeline end-to-end có lưu segment để đo sync
!python evaluation/benchmark_perf.py --mode end_to_end --keep-temp-segments --include-nvidia-smi

## 8. Chạy Benchmark 3: Group-level Sync
Đo sự chênh lệch (MAE overflow) sau khi áp dụng thuật toán gom nhóm phụ đề.

In [ ]:
!python evaluation/benchmark_sync_group.py

## 9. Xuất kết quả về Google Drive
Gói toàn bộ kết quả benchmark ném sang Drive của bạn để đưa vào CV hoặc Report.

In [ ]:
import shutil
import os

DRIVE_DIR = "/content/drive/MyDrive/VidTranscribe_Evaluation"
os.makedirs(DRIVE_DIR, exist_ok=True)

files_to_copy = [
    "evaluation/localization_results.csv",
    "evaluation/localization_results.json",
    "evaluation/sync_group_results.csv",
    "evaluation/sync_group_results.json",
    "evaluation/perf_results.json"
]

for f in files_to_copy:
    if os.path.exists(f):
        shutil.copy(f, DRIVE_DIR)
        print(f"Đã copy {f} -> {DRIVE_DIR}")
    else:
        print(f"Không tìm thấy {f}")

print("\nHoàn tất toàn bộ quy trình! Kiểm tra Google Drive của bạn.")